# Lead Scoring Pipeline

This notebook scores leads using an XGBoost model.

## Model
- Algorithm: XGBoost
- Target: Lead conversion probability

In [ ]:
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
%%time
currdt = datetime.date(2023,1,31)
data_check = pd.DataFrame({'status': ['ok']})

In [ ]:
%matplotlib inline
plt.style.use('ggplot')
print("Plotting configured")

In [ ]:
%%sql
SELECT customer_id, lead_score, created_at
FROM sch_demo.table_leads
WHERE created_at >= '2023-01-01'
ORDER BY lead_score DESC
LIMIT 100

In [ ]:

jdbcUrl_01 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_01".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_02 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_02".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_03 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_03".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_04 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_04".format("demo001.corp.example.com", 1433, "db_demo")
connectionProperties = {
  "user" : "",
  "password" : ""}# need to replace

print(" Demo Scoring Script is in Execution")
print(" -------------------------------------------")
print(" Demo Input sheet is for: ",currdt.strftime('%d-%b-%Y').upper() )

## Paths ###
base_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline' 
dic_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries'
Log_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline/logs'



In [ ]:
### Logging scripts after each print statements ###
###################################################
import pytz
IST = pytz.timezone('Asia/Kolkata')
batch_start = datetime.datetime.now(IST)
script_log= []
log_1 = '### Demo Scoring Log Report ### :\n\n1. Script Started at: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
import pickle
filename = r'/dbfs/mnt/demo/dl_demo_ds/pipeline/model/DEMO_XGB_MODEL.sav'

# ####### load the model from disk
xgbtc = pickle.load(open(filename, 'rb'))


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n2. Model loaded: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
############# read input data 
filename = 'Demo_Leads_' + str(currdt.strftime('%d%b%Y').upper()) + '.csv'
lead_df_FY22 = pd.read_csv(base_path +'/Demo Input/' +filename)
lead_df_FY22.fillna(0,inplace=True)
score_data = lead_df_FY22[['CGI','proj_sales_3mon','num_funds_sales','max_aum_2years','proj_sales','rollup_12_24monsales','fund_high_Returns','Fund_Returns','aum_0','fund_med_Returns','Sales_GT_100K_2years','rollup_12monredemptions','rollup_12_24monredemptions','nsales_segment','M']]
score_data.rename(columns={'aum_0':'aumsep20'},inplace=True)
one_hot = pd.get_dummies(score_data['nsales_segment'])
score_data = score_data.join(one_hot)
del score_data['Holding Asset']
del score_data['nsales_segment']

############# scoring data
lead_fy22_output = pd.concat([score_data.reset_index(drop=True),pd.DataFrame(xgbtc.predict_proba(score_data.iloc[:,1:])[:,1],columns = ['xgb_Prob'])], axis=1)
lead_fy22_output = pd.merge(lead_fy22_output,lead_df_FY22[['CGI','rollup_12monsales']],how='left',on='CGI')
lead_fy22_output.fillna(0,inplace=True)
lead_fy22_output.rename(columns={'aumsep20':'aum_0'},inplace=True)



## Scoring Logic

Apply model predictions and business rules:
- XGBoost probability
- IA Score calculation
- Blanket rules for high-value leads

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n3. input sheet loaded and sored: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
for i in ['proj_sales_3mon','aum_0','proj_sales','rollup_12monsales']:
  lead_fy22_output[i] = np.where(lead_fy22_output[i]==0,np.nan,lead_fy22_output[i])


lead_fy22_output['AUM_decile']= lead_fy22_output.aum_0.rank(pct=True,na_option='keep').fillna(0)
lead_fy22_output['Sales_decile']= lead_fy22_output.rollup_12monsales.rank(pct=True,na_option='keep').fillna(0)
lead_fy22_output['Proj_sales3m_decile']= lead_fy22_output.proj_sales_3mon.rank(pct=True,na_option='keep').fillna(0)
lead_fy22_output['Proj_sales_decile']= lead_fy22_output.proj_sales.rank(pct=True,na_option='keep').fillna(0)
################################################################################################################

lead_fy22_output.fillna(0,inplace=True)


##################### ia scores for Defend and others ####
lead_fy22_output['IA_Score'] = round((lead_fy22_output['Proj_sales3m_decile']+lead_fy22_output['Proj_sales_decile']+lead_fy22_output['Sales_decile']+ lead_fy22_output['AUM_decile'])/4,3)

lead_fy22_output['Blanket_Rules'] = np.where(((lead_fy22_output['aum_0']>=1000000)|(lead_fy22_output['rollup_12monsales']>=100000)|(lead_fy22_output['proj_sales_3mon']>=100000)|(lead_fy22_output['proj_sales']>=100000)),1,0)

lead_fy22_output['financial'] = np.where(((lead_fy22_output['aum_0']>=1000000)|(lead_fy22_output['rollup_12monsales']>=100000)|(lead_fy22_output['proj_sales_3mon']>=100000)|(lead_fy22_output['proj_sales']>=100000)),1,0)

lead_fy22_output['Final_Score'] = np.where((lead_fy22_output['Blanket_Rules']==1)&(lead_fy22_output['xgb_Prob']<0.225),0.225+0.01*lead_fy22_output['IA_Score'],lead_fy22_output['xgb_Prob'])

lead_fy22_output['flag']=1
lead_fy22_output = lead_fy22_output.sort_values("xgb_Prob",ascending=False).reset_index(drop=True)
lead_fy22_output['Model_Rank'] = lead_fy22_output.flag.cumsum()
del lead_fy22_output['flag']

lead_fy22_output['Lead_Indicator'] = np.where((lead_fy22_output['Model_Rank']<=600)|(lead_fy22_output['Blanket_Rules']==1),1,0)

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n4. IA score assigned: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
###################### to exclusion list add three ids
pushdown_query = "(SELECT distinct(GlobalID) from sch_demo_02.table_01) temp" 
exclude2 = spark.read.jdbc(url=jdbcUrl_02, table=pushdown_query, properties=connectionProperties)
exclude2 = exclude2.toPandas()
exclude2.columns = ['contact_global_id_exclude']
# exclude2['contact_global_id_exclude'] = exclude2['contact_global_id_exclude'].astype(int)
exclude2['contact_global_id_exclude'] = exclude2['contact_global_id_exclude'].astype(str)
exclude2['exclude'] = 1

pushdown_query = "(SELECT distinct([Contact.Global Id]) from sch_demo_03.table_02) temp" 
exclude1 = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
exclude1 = exclude1.toPandas()
exclude1.columns = ['contact_global_id_exclude']
exclude1['contact_global_id_exclude'] = exclude1['contact_global_id_exclude'].astype(str)
exclude1['exclude'] = 1

exclude = pd.concat([exclude1,exclude2],axis=0)
exclude.reset_index(inplace=True,drop=True)
exclude.loc[len(exclude.index)] = [1000001,1]
exclude.loc[len(exclude.index)] = [1000002,1]
exclude.loc[len(exclude.index)] = [1000003,1]
exclude.rename(columns={'contact_global_id_exclude':'Contact_Global_ID'},inplace=True)

################ to house accounts add two ids
pushdown_query = "(SELECT * from sch_demo_03.table_03) temp" 
house_accounts = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
house_accounts = house_accounts.toPandas()
house_accounts = pd.DataFrame(house_accounts['CONTACT_GLOBAL_ID'].unique(),columns=['CONTACT_GLOBAL_ID'])
house_accounts['ind'] = 1
house_accounts.reset_index(inplace=True,drop=True)
house_accounts.loc[len(house_accounts.index)] = [1000004,1]
# house_accounts.loc[len(house_accounts.index)] = [1000005,1]
house_accounts['CONTACT_GLOBAL_ID']=house_accounts['CONTACT_GLOBAL_ID'].astype('str')
house_accounts.rename(columns={'CONTACT_GLOBAL_ID':'Contact_Global_ID'},inplace=True)



In [ ]:
def remove_liq_funds(df):
  df['FUND_NAME1'] = df.FUND_NAME.str.lower()
  df['FUND_NAME2'] = df.FUND_NAME1.str.contains('money|liquid', regex=True)
  df = df.loc[df.FUND_NAME2 == False,] 
  df.drop(['FUND_NAME1','FUND_NAME2'],axis=1,inplace=True)
  return df

def remove_exclusion_ids(df):
  df = pd.merge(df,exclude,on= 'Contact_Global_ID',how='left')
  df = df[df['exclude'] != 1.0]
  del df['exclude']
  return df

def remove_house_accounts(df):
  df = pd.merge(df,house_accounts,how='left',on='Contact_Global_ID')
  df = df[df['ind'] != 1]
  del df['ind']
  return df

def remove_unmapped_ids(df):
  df['len'] = df['Contact_Global_ID'].apply(lambda x:len(x))
  df = df[df['len'].isin([7,8])]
  del df['len']
  df.reset_index(inplace=True,drop=True)
  return df

pushdown_query = "(SELECT * from sch_demo_03.table_04) temp" 
crm_contacts = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
crm_contacts = crm_contacts.toPandas()
crm_contacts = crm_contacts[crm_contacts.Date_inserted== crm_contacts.Date_inserted.max()]
crm_contacts=crm_contacts[crm_contacts['Primary Owner'].isin(['Surname01, Name01','Surname02, Name02','Surname03, Name03'])]
crm_contacts = crm_contacts[['Contact ID','Primary Owner']]
crm_contacts.columns = ['Contact_Global_ID','PSP']
crm_contacts.Contact_Global_ID=crm_contacts.Contact_Global_ID.astype('str')

crm_contacts = remove_exclusion_ids(crm_contacts)

crm_contacts = remove_house_accounts(crm_contacts)



In [ ]:
crm_adv_contacts = crm_contacts.copy()
crm_adv_contacts.PSP = np.where(crm_adv_contacts.PSP == 'Name01 Surname01', 'Surname01, Name01',np.where(crm_adv_contacts.PSP == 'Name02 Surname02', 'Surname02, Name02',np.where(crm_adv_contacts.PSP == 'Name03 Surname03', 'Surname03, Name03',crm_adv_contacts.PSP)))
crm_adv_contacts.columns = ['CGI','PSP']
crm_adv_contacts['CGI'] = crm_adv_contacts['CGI'].astype(int)


In [ ]:
lead_fy22_output1 = pd.merge(lead_fy22_output, crm_adv_contacts, how='left',on='CGI')


lead_fy22_output_origialrank = lead_fy22_output1[['CGI','IA_Score','PSP']].copy()
lead_fy22_output_origialrank = lead_fy22_output_origialrank.sort_values(by=['PSP','IA_Score'],ascending=[True,False])
lead_fy22_output_origialrank.reset_index(inplace=True,drop=True)
lead_fy22_output_origialrank['original_rank'] = lead_fy22_output_origialrank.groupby('PSP').IA_Score.rank(method='first',na_option='bottom',ascending=False)


In [ ]:
#################################### remove the ownership list from final dataframe
pushdown_query = "(select * from sch_demo_03.table_05) temp" 
ownership = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
ownership = ownership.toPandas() ####
ownership = ownership[(ownership['ownership_incremental'] == 'ownership')]
ownership = ownership[['Contact Global ID','New_PSP']]
ownership.columns = ['CGI','final_psp']
ownership.CGI = ownership.CGI.astype('int')

#remove ownership leads
lead_fy22_output_ownership = lead_fy22_output1[lead_fy22_output1['CGI'].isin(ownership['CGI'])]
lead_fy22_output_ownership = pd.merge(lead_fy22_output_ownership,ownership,how='left',on='CGI')
lead_fy22_output_ownership['own_inc'] = 'ownership'


#for EW's check if any psp changed then replace with current psp
for i in range(len(lead_fy22_output_ownership)):
  if lead_fy22_output_ownership.loc[i,'PSP'] == 'Surname02, Name02' and lead_fy22_output_ownership.loc[i,'final_psp'] != 'Name04 Surname04':
    lead_fy22_output_ownership.loc[i,'final_psp'] = lead_fy22_output_ownership.loc[i,'PSP']
  if lead_fy22_output_ownership.loc[i,'PSP'] == 'Surname03, Name03' and lead_fy22_output_ownership.loc[i,'final_psp'] != 'Name04 Surname04':
    lead_fy22_output_ownership.loc[i,'final_psp'] = lead_fy22_output_ownership.loc[i,'PSP']
  if lead_fy22_output_ownership.loc[i,'PSP'] == 'Surname01, Name01' and lead_fy22_output_ownership.loc[i,'final_psp'] != 'Name04 Surname04':
    lead_fy22_output_ownership.loc[i,'final_psp'] = lead_fy22_output_ownership.loc[i,'PSP']

    
lead_fy22_output_ownership.final_psp = np.where(lead_fy22_output_ownership.final_psp.isin(['Name04 Surname04']),'Surname04,Name04' ,lead_fy22_output_ownership.final_psp )


In [ ]:
lead_fy22_output_nonownership = lead_fy22_output1[~lead_fy22_output1['CGI'].isin(ownership['CGI'])]


##########  get incremental leads from non ownership by re-ranking non ownership leads and selecting top 25 leads with financial indicator
lead_fy22_output_nonownershiprank = lead_fy22_output_nonownership[['PSP','IA_Score','CGI']]

lead_fy22_output_nonownershiprank = lead_fy22_output_nonownershiprank.sort_values(by=['PSP','IA_Score'],ascending=[True,False])
lead_fy22_output_nonownershiprank.reset_index(inplace=True,drop=True)
lead_fy22_output_nonownershiprank['rank'] = lead_fy22_output_nonownershiprank.groupby('PSP').IA_Score.rank(method='first',na_option='bottom',ascending=False)


lead_fy22_output_nonownership = pd.merge(lead_fy22_output_nonownership,lead_fy22_output_nonownershiprank[['CGI','rank']],how='left')
lead_fy22_output_nonownership['own_inc'] = ''
lead_fy22_output_nonownership['final_psp'] = ''

################### add or condition with probability >= 0.225 along with financial =1
for i in range(len(lead_fy22_output_nonownership)):
  if lead_fy22_output_nonownership.loc[i,'Lead_Indicator'] == 1 and lead_fy22_output_nonownership.loc[i,'rank'] <= 25:
    lead_fy22_output_nonownership.loc[i,'own_inc'] = 'incremental'
    lead_fy22_output_nonownership.loc[i,'final_psp'] = lead_fy22_output_nonownership.loc[i,'PSP']
  elif lead_fy22_output_nonownership.loc[i,'Lead_Indicator'] == 1 and lead_fy22_output_nonownership.loc[i,'rank'] > 25 and lead_fy22_output_nonownership.loc[i,'rank'] <= 35:
    lead_fy22_output_nonownership.loc[i,'own_inc'] = 'incremental'
    lead_fy22_output_nonownership.loc[i,'final_psp'] = 'Surname04,Name04'
  else:
    lead_fy22_output_nonownership.loc[i,'own_inc'] = 'non lead'
    lead_fy22_output_nonownership.loc[i,'final_psp'] = lead_fy22_output_nonownership.loc[i,'PSP']

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n5. incremental leads identified from non ownership advisors: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
############## combine ownership and non ownership and rank based on IAscore
final_output = pd.concat([lead_fy22_output_ownership,lead_fy22_output_nonownership[lead_fy22_output_ownership.columns]],axis=0)
final_output.reset_index(inplace=True,drop=True)

#rerank all advisor as per EW's, increase ownership/incremental rank by 1
final_output1 = final_output[['CGI','IA_Score','PSP','own_inc']].copy()
final_output1['IA_Score'] = np.where((final_output1['own_inc'] == 'ownership') | (final_output1['own_inc'] == 'incremental'),final_output1['IA_Score']+1,final_output1['IA_Score'])
final_output1 = final_output1.sort_values(by=['PSP','IA_Score'],ascending=[True,False])
final_output1['original_rank'] = final_output1.groupby('PSP').IA_Score.rank(method='first',na_option='bottom',ascending=False)

final_output = pd.merge(final_output,final_output1[['CGI','original_rank']],how='left',on='CGI')

In [ ]:
################ lapsed advisor logic
final_output['fallen_angel'] = np.where((final_output.rollup_12monsales <= 0)&(final_output.aum_0 >= 500000),'Fallen Angel','Active')
final_output['own_inc'] = np.where((final_output['own_inc']=='non lead')&(final_output['fallen_angel'] == 'Fallen Angel'),'incremental',final_output['own_inc'])

################# cream advisor logic
final_output['own_inc'] = np.where((final_output['own_inc']=='non lead')&(final_output['Blanket_Rules'] == 1), 'incremental', final_output['own_inc'])
final_output['Type'] = 'Entity'

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n6. final rankorder and fallen angels identified: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Scoring_Logs.txt",index=False,columns=None)

In [ ]:
filename = "Demo_Leads_"+currdt.strftime('%d%b%Y').upper()+".csv"
final_output.to_csv(base_path+"/Demo Output/"+filename,index=False) 
print("Scored sheet ready - ProgressBar:100%")